# 1. Importación de librerías y configuración inicial
En esta sección se importaran las librerías necesarias para el procesamiento, análisis y visualización de los datos.
Sugerencias: pandas, numpy, matplotlib, folium

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import folium

print('Librerias cargadas correctamente')
print(f'Version de pandas: {pd.__version__}')
print(f'Version de numpy: {np.__version__}')
print(f'Version de folium: {folium.__version__}')

Librerias cargadas correctamente
Version de pandas: 2.2.2
Version de numpy: 2.0.2
Version de folium: 0.20.0


# 2. Carga de datos


*   Carga de base de datos IPASH Suelo
*   Carga de base de datos IPASH Agua
*   Carga de base de datos IPASH Sedimento
*   Carga de normativas
*   Carga de datos para cobertura vegetal: MapBiomas + GEE (NDVI)







In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#Componente suelo:
#URL de exportación: https://docs.google.com/spreadsheets/d/17zSeu4qP2TIJK7dy0MWKBqZOAkq24xCq/edit?usp=drive_link&ouid=110342148515634196231&rtpof=true&sd=true
URL_SUELO = 'https://docs.google.com/spreadsheets/d/17zSeu4qP2TIJK7dy0MWKBqZOAkq24xCq/export?format=xlsx'
suelo = pd.read_excel(URL_SUELO)
suelo.head()

In [ ]:
#Componente agua
#URL de exportación: https://docs.google.com/spreadsheets/d/1sf285A4XFBtwktd6E2X1ZmDnr1OJaTv6/edit?usp=drive_link&ouid=110342148515634196231&rtpof=true&sd=true
URL_AGUA = 'https://docs.google.com/spreadsheets/d/1sf285A4XFBtwktd6E2X1ZmDnr1OJaTv6/export?format=xlsx'
agua = pd.read_excel(URL_AGUA)
agua.head()

In [ ]:
#Componente sedimento
#URL de exportación: https://docs.google.com/spreadsheets/d/1QguQUQ66N5LRcYsGiQuEwuIf9XOrxTbF/edit?usp=drive_link&ouid=110342148515634196231&rtpof=true&sd=true
import warnings
warnings.filterwarnings('ignore')
URL_SEDIMENTO = 'https://docs.google.com/spreadsheets/d/1QguQUQ66N5LRcYsGiQuEwuIf9XOrxTbF/export?format=xlsx'
sedimento = pd.read_excel(URL_SEDIMENTO)
sedimento.head()


## Carga de Normativas Ambientales

Para la evaluación del cumplimiento ambiental en las distintas matrices de estudio (agua, suelo y sedimentos) se consideraron normativas vigentes tanto en Perú como en Argentina.
En el caso de **Perú**, se trabajó con los *Estándares de Calidad Ambiental* (ECA) para agua y suelo (Decretos Supremos N° 004-2017-MINAM y N° 011-2017-MINAM, respectivamente), así como con los *Límites Máximos Permisibles* (LMP) aplicables a efluentes líquidos.

Para **Argentina**, la normativa ambiental utilizada como referencia (Ley Nacional de Residuos Peligrosos N°24.051 y su Decreto Reglamentario N°831/93), no establece límites de calidad explícitos por matriz de la misma manera que los ECA en Perú. Sin embargo, presenta criterios para la identificación y gestión de residuos peligrosos, incluyendo concentraciones de referencia para ciertos contaminantes.
Dado que estos valores no constituyen estándares de calidad ambiental directamente comparables, en el presente trabajo se emprlean como referencia complementaria, junto con criterios técnico-interpretativos y valores guía utilizados en la práctica, a fin de evaluar el grado de contaminación y permitir una comparación entre ambos marcos regulatorios.

- Normativa para Suelo (Perú): https://sinia.minam.gob.pe/normas/aprueban-estandares-calidad-ambiental-eca-suelo

- Normativa para Agua (Perú): https://www.minam.gob.pe/wp-content/uploads/2017/06/DS-004-2017-MINAM.pdf

- Límites Máximos Permisibles (Perú): https://sinia.minam.gob.pe/normas/limites-maximos-permisibles-lmp-efluentes-liquidos-sub-sector

- Normativa para Suelo (Decreto 831/93, Tabla 9, Argentina):https://servicios.infoleg.gob.ar/infolegInternet/anexos/10000-14999/12830/norma.htm

- Normativa para Agua (Decreto 831/93, Tabla 5, Argentina): https://servicios.infoleg.gob.ar/infolegInternet/anexos/10000-14999/12830/norma.htm



In [ ]:
# Normativas Perú
#URL de exportación: https://docs.google.com/spreadsheets/d/16_k1zxeZNGY96GGzMfrbVDdiSNciSNk-/edit?usp=drive_link&ouid=110342148515634196231&rtpof=true&sd=true
URL_NORM_PERU = 'https://docs.google.com/spreadsheets/d/16_k1zxeZNGY96GGzMfrbVDdiSNciSNk-/export?format=xlsx'
norm_peru = pd.read_excel(URL_NORM_PERU)
norm_peru.head()

In [ ]:
# Normativas Argentina
# URL de exportación: https://docs.google.com/spreadsheets/d/1tj6sqxXjmq_hBnYca8EnNQtYZvD9P05G/edit?usp=drive_link&ouid=110342148515634196231&rtpof=true&sd=true
URL_NORM_ARG = 'https://docs.google.com/spreadsheets/d/1tj6sqxXjmq_hBnYca8EnNQtYZvD9P05G/export?format=xlsx'
norm_arg = pd.read_excel(URL_NORM_ARG)
norm_arg.head()

# 3. Exploración, Limpieza y Validacion de datos


*   Revisión de nombres columnas ✅
*   Conversión numérica / fecha ✅
*   Cantidad de monitoreos por matriz y puntos de muestreo ✅
*   Periodo temporal del cada monitoreo (por matriz) ✅
*   Zona geográfica de muestreo (UTM) ✅
*   Convertir unidades de UTM al sistema WGS84 (EPSG:4326) ✅
*   Mapa de puntos de monitoreo por matriz ✅   
*   Revisión/Corrección de coordenadas en puntos de muestreo ❌
*   Tratamiento de valores faltantes ❌
*   Tratamiento de datos bajo el limite de detección ❌
*   Detección de outliers ❌
*   Exclusión o corrección documentada ❌

***Recordar dejar una celda de texto explicando las decisiones tomadas***









## 3.1 Exploración de datos

In [ ]:
## Exploración de datos: revisar nombre de columnas
suelo.columns

In [ ]:
agua.columns

In [ ]:
sedimento.columns

**Mini-conclusión:** Los 3 set de datos comparten todas las columnas salvo "Etapa de extracción secuencial" que solo se encuentra en suelos y sedimentos.

In [ ]:
## Exploración de datos: Convertir fecha a datetime

suelo['Fecha'] = pd.to_datetime(suelo['Fecha'])
suelo.head(10)


In [ ]:
agua['Fecha'] = pd.to_datetime(agua['Fecha'])
agua.head(10)

In [ ]:
sedimento['Fecha'] = pd.to_datetime(sedimento['Fecha'])
sedimento.head(10)

In [ ]:
## Exploración de datos: Cantidad de monitoreos por matriz
suelo['Etapa'].value_counts()



In [ ]:
agua['Etapa'].value_counts()

In [ ]:
sedimento['Etapa'].value_counts()

In [ ]:
## Exploración de datos: cantidad de puntos por monitoreo (utilizando localizacion geoespacial)

In [ ]:
def agregar_id_punto(df):
  return (
      df['Este'].astype(str) + '_' +
      df['Norte'].astype(str) + '_' +
      df['Zona'].astype(str)
  )

In [ ]:
suelo['id_punto_geografico'] = agregar_id_punto(suelo)
agua['id_punto_geografico'] = agregar_id_punto(agua)
sedimento['id_punto_geografico'] = agregar_id_punto(sedimento)

In [ ]:
def puntos_por_monitoreo(df):
  return df.groupby('Etapa')['id_punto_geografico'].nunique()

puntos_por_monitoreo(suelo)



In [ ]:
puntos_por_monitoreo(agua)

In [ ]:
puntos_por_monitoreo(sedimento)

In [ ]:
## Exploración de datos: Periodo temporal por monitoreo
def resumen_temporal(df):
  return df.groupby('Etapa')['Fecha'].agg(['min', 'max'])

resumen_temporal(suelo)

In [ ]:
resumen_temporal(agua)

In [ ]:
resumen_temporal(sedimento)

**Mini conclusión**:
- Para suelo hay 2 campañas de monitoreo: Ago-14 hasta Oct-19 (711) y Ene-19 hasta Sep-19 (288)
- Para agua hay dos campañas de monitoreo: Oct-13 hasta Jun-19 (722) y Jul-14 hasta Ene-19 (276)
- Para sedimento solo hay una campaña desde Oct-13 hasta Oct-19 (218 parámetros)

In [ ]:
## Exploración de datos: Zona geográfica donde se tomaron las muestras
print('Zona Geográfica de Muestreo (UTM): Matriz Suelo')
print(suelo['Zona'].value_counts())

In [ ]:
print('Zona Geográfica de Muestreo (UTM): Matriz Agua')
print(agua['Zona'].value_counts())

In [ ]:
print('Zona Geográfica de Muestreo (UTM): Matriz Sedimento')
print(sedimento['Zona'].value_counts())

In [ ]:
## Exploracion de datos: Verificar 'unidades' de las coordenadas geográficas
suelo['Zona'].unique()

In [ ]:
## Exploracion de datos: Convertir unidades de UTM al sistema WGS84 (EPSG:4326)
!pip install pyproj

In [ ]:
##Exploración de datos: mapa de puntos de monitoreo por matriz

In [ ]:
# Conversión de Unidades

from pyproj import Transformer

def utm_to_latlon_dynamic(row):
    # Determinar el sistema de referencias de coordenadas (CRS) UTM a partir de la columna 'Zona'
    # Suponiento el Hemisferio Sur para Perú (EPSG:327XX)
    utm_crs = f"epsg:327{int(row['Zona'])}" #Como las muestras pueden venir de diferentes UTM, esta función genera el EPSG dinámico
    # Transformer.from_crs expects source_crs, target_crs
    # always_xy=True ensures output is (longitude, latitude)
    transformer = Transformer.from_crs(utm_crs, "epsg:4326", always_xy=True)

    # 'Este' is Easting, 'Norte' is Northing
    lon_wgs84, lat_wgs84 = transformer.transform(row['Este'], row['Norte'])
    return pd.Series([lat_wgs84, lon_wgs84])

In [ ]:
# Estas lineas son para aplicar la conversión dinámica al DataFrame.
# Despues hay que asegurarse que las columnas de 'Zona', 'Este', 'Norte' sean numéricas y esta linea gestiona posibles errores
for df in [suelo, agua, sedimento]:
    for col in ['Este', 'Norte', 'Zona']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df.dropna(subset=['Este', 'Norte', 'Zona'], inplace=True)

suelo[['lat_wgs84', 'lon_wgs84']] = suelo.apply(utm_to_latlon_dynamic, axis=1) #Esta linea crea las coordenadas en el formato que son necesarias para Folium (suelo)
agua[['lat_wgs84', 'lon_wgs84']] = agua.apply(utm_to_latlon_dynamic, axis=1) #Esta linea crea las coordenadas en el formato  que son necesarias para Folium (agua)
sedimento[['lat_wgs84', 'lon_wgs84']] = sedimento.apply(utm_to_latlon_dynamic, axis=1) #Esta linea crea las coordenadas en el formato  que son necesarias para Folium (sedimento)

print("Coordenadas convertidas a WGS84 para suelo, agua y sedimento.")

# Display the head of 'agua' with new coordinates for verification
display(agua[['Nombre del punto', 'Este', 'Norte', 'Zona', 'lat_wgs84', 'lon_wgs84']].head())

In [ ]:
#Exploración de datos: cuantos puntos unicos de muestreo tiene cada matriz?
print('Suelo:', suelo['id_punto_geografico'].nunique())
print('Agua:', agua['id_punto_geografico'].nunique())
print('Sedimento:', sedimento['id_punto_geografico'].nunique())

## 3.2 Primera visualizadción de los puntos de monitoreo

In [ ]:
## OBJETIVO: Hacer mapa de puntos de monitoreo por matriz (suelo)

###Primero: eliminar duplicados
df_plot = suelo.drop_duplicates(subset=['id_punto_geografico', 'Etapa'])

###Segundo: armar mapa
mapa_suelo = folium.Map(location=[df_plot['lat_wgs84'].mean(), df_plot['lon_wgs84'].mean()], zoom_start=10)

###Tercero: agregar puntos
for _, row in df_plot.iterrows():
  color = ('steelblue' if row['Etapa'] == 'Primer monitoreo' else 'darkred')
  folium.CircleMarker(
      location=[row['lat_wgs84'], row['lon_wgs84']],
      radius=2,
      color=color,
      fill=True,
      fill_opacity=0.7,
      popup=row['Nombre del punto']
  ).add_to(mapa_suelo)

###Cuarto: se le pone nombre al mapa
title_html = '''
             <h3 align="center" style="font-size:16px"><b>Puntos de monitoreo - Suelo</b></h3>
             '''
mapa_suelo.get_root().html.add_child(folium.Element(title_html))

###Quinto: agregar leyenda en el mapa

mapa_suelo

🚧🚧🚧🚧🚧🚧🚧🚧🚧🚧🚧🚧
**OJO!!! APARECIÓ EL PRIMER PUNTO DE MONITOREO POR FUERA DEL ÁREA CONTINENTAL --> HAY QUE ELIMNARLO**
esta a la altura de la ciudad de Chiclayo
Hay que limpiar el DataSet

In [ ]:
## OBJETIVO: Hacer mapa de puntos de monitoreo por matriz (agua)

###Primero: eliminar duplicados
df_plot = agua.drop_duplicates(subset=['id_punto_geografico', 'Etapa'])

###Segundo: armar mapa
mapa_agua = folium.Map(location=[df_plot['lat_wgs84'].mean(), df_plot['lon_wgs84'].mean()], zoom_start=10)

###Tercero: agregar puntos
for _, row in df_plot.iterrows():
  color = ('steelblue' if row['Etapa'] == 'Primer monitoreo' else 'darkred')
  folium.CircleMarker(
      location=[row['lat_wgs84'], row['lon_wgs84']],
      radius=2,
      color=color,
      fill=True,
      fill_opacity=0.7,
      popup=f'{row['Nombre del punto']}<br>{row["Etapa"]}'
  ).add_to(mapa_agua)

###Cuarto: se le pone nombre al mapa
title_html = '''
             <h3 align="center" style="font-size:16px"><b>Puntos de monitoreo - Agua</b></h3>
             '''
mapa_agua.get_root().html.add_child(folium.Element(title_html))

###Quinto: agregar leyenda en el mapa

mapa_agua

🚧🚧🚧🚧🚧🚧🚧🚧🚧🚧🚧🚧
**OJO!!! APARECIERON DOS PUNTOS DEL PRIMER MONITOREO POR FUERA DEL ÁREA CONTINENTAL --> HAY QUE ELIMNARLOS**
Hay que limpiar el DataSet --> estan por debajo de lima

In [ ]:
## OBJETIVO: Hacer mapa de puntos de monitoreo por matriz (sedimento)

###Primero: eliminar duplicados
df_plot = sedimento.drop_duplicates(subset=['id_punto_geografico', 'Etapa'])

###Segundo: armar mapa
mapa_sedimento = folium.Map(location=[df_plot['lat_wgs84'].mean(), df_plot['lon_wgs84'].mean()], zoom_start=10)

###Tercero: agregar puntos
for _, row in df_plot.iterrows():
  color = ('steelblue' if row['Etapa'] == 'Primer monitoreo' else 'darkred')
  folium.CircleMarker(
      location=[row['lat_wgs84'], row['lon_wgs84']],
      radius=2,
      color=color,
      fill=True,
      fill_opacity=0.7,
      popup=f'{row['Nombre del punto']}<br>{row["Etapa"]}'
  ).add_to(mapa_sedimento)

###Cuarto: se le pone nombre al mapa
title_html = '''
             <h3 align="center" style="font-size:16px"><b>Puntos de monitoreo - Sedimento</b></h3>
             '''
mapa_sedimento.get_root().html.add_child(folium.Element(title_html))

###Quinto: agregar leyenda en el mapa

mapa_sedimento

## 3.3 Limpieza de datos

In [ ]:
## Limpieza de datos: Corrección de coordenadas en puntos de muestreo (FALTA HACER)

In [ ]:
## Limpieza de datos: Tratamiento de valores faltantes

#Con esta linea de códigos detectamos columnas con datos faltantes
suelo_nulos = suelo.isnull().sum()
print('Valores nulos por columna: Matriz Suelo')
print(suelo_nulos[suelo_nulos > 0] if suelo_nulos.any() else 'No hay valores nulos')

In [ ]:
agua_nulos = agua.isnull().sum()
print('Valores nulos por columna: Matriz Agua')
print(agua_nulos[agua_nulos > 0] if agua_nulos.any() else 'No hay valores nulos')

In [ ]:
sedimento_nulos = sedimento.isnull().sum()
print('Valores nulos por columna: Matriz Sedimento')
print(sedimento_nulos[sedimento_nulos > 0] if sedimento_nulos.any() else 'No hay valores nulos')

**MINI-CONCLUSIÓN:** Solo se registraron valores nulos para el dataset de suelo, en las columnas de tipo de análisis y etapa secuencial de extracción.
Para el caso de los dataset de agua y sedimento, todas las celdas contienen información

In [ ]:
## Limpieza de datos: Tratamiento de valores por debajo del límite de detección

#Primero: 'miramos' cuantos valores tienen el símbolo '<' en la celda
tiene_menor_suelo = suelo['Valor'].astype(str).str.contains('<')
print(f'Valores debajo el límite de detección (Suelo): {tiene_menor_suelo.sum()} de {len(suelo)} ({tiene_menor_suelo.mean()*100:.1f}%')
tiene_menor_agua = agua['Valor'].astype(str).str.contains('<')
print(f'Valores debajo el límite de detección (Agua): {tiene_menor_agua.sum()} de {len(agua)} ({tiene_menor_agua.mean()*100:.1f}%')
tiene_menor_sedimento = sedimento['Valor'].astype(str).str.contains('<')
print(f'Valores debajo el límite de detección (Sedimento): {tiene_menor_sedimento.sum()} de {len(sedimento)} ({tiene_menor_sedimento.mean()*100:.1f}%)')

In [ ]:
#Segundo: limpiamos los valores y lo reemplazamos por LD/2
import re
def limpiar_valor(v):
  '''
  Convierte un valor analítico a numero_
  - Si contiene '<', extra el LD y lo divide entre 2 (convención USEPA)
  - Reemplaza coma decimal por punto
  - Si no puede convertirse, devuelve NaN
  '''

  v = str(v).strip()
  v = v.replace(',','.')     #linea para cambiar la coma decimal por el punto
  if v.startswith('<'):
    try:
      lod = float(v.replace('<', '').strip())   #linea para extraer el número despues del '<'
      return lod/2
    except ValueError:
      return np.nan
  try:
    return float(v)
  except ValueError:
    return np.nan

#Tercero: aplicar la función a toda la columna Valor de cada dataset
suelo['Valor_num'] = suelo['Valor'].apply(limpiar_valor)

#Cuarto: imprimir resultado
print('Columna Valor_num creada')
print(f'Valores numéricos válidos (suelo): {suelo['Valor_num'].notna().sum}')
print(f'Valores no convertibles (NaN) (suelo): {suelo['Valor_num'].isna().sum}')

In [ ]:
agua['Valor_num'] = agua['Valor'].apply(limpiar_valor)
print(f'Valores numéricos válidos (agua): {agua['Valor_num'].notna().sum}')
print(f'Valores no convertibles (NaN) (agua): {agua['Valor_num'].isna().sum}')

In [ ]:
sedimento['Valor_num'] = sedimento['Valor'].apply(limpiar_valor)
print(f'Valores numéricos válidos (sedimento): {sedimento['Valor_num'].notna().sum}')
print(f'Valores no convertibles (NaN) (sedimento): {sedimento['Valor_num'].isna().sum}')

In [ ]:
# Quinto: Comparar el correcto agregado de la columna
sedimento[['Parámetro', 'Valor', 'Valor_num', 'Unidad de medida']].head(10)

#Quedó el comando con sedimento pero corroboré con los 3 dataset

## 3.4 Validación de datos
esto no se si se va a hacer aca o en el análisis exploratorio

In [ ]:
## Validación de datos: Detección de outliers

In [ ]:
## Validación de datos: Exclusión o corrección documentada de datos

# 4. Análisis exploratorio de datos (EDA)

*   Resumen de la Estructura del dataset ⏳
*   Parámetros analizados ✅
*   Estadística descriptica ⏳
*   Distribución de HTP y otros contaminantes
*   Comparación entre matrices







## 4.1 Inspección General

In [ ]:
### PRUEBA PARA MAPA INTERACTIVO: LA IDEA ES CONSTRUIR UN MAPA INTERACTIVO DONDE
#APAREZCAN LOS DATOS DEL PRIMER MONITOREO Y EN UN DESPLEGABLE PUEDA IR CAMBIANDO
#LA VISUALIZACION POR CONTAMINANTE. SE PROBARÁ CON 3: HTP TOTALES, PAH TOTALES Y
#METALES TOTALES. LA IDEA ES QUE TENGA CADA PUNTO GEOGRÁFICO UN COLOR SEGUN LA
#CONCENTRACION

In [ ]:
#Primero: importar Ploly
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
print('suelo:', 'Valor_num' in suelo.columns, suelo['Valor_num'].dtype)
print('agua:', 'Valor_num' in agua.columns, agua['Valor_num'].dtype)
print('sedimento:', 'Valor_num' in sedimento.columns, sedimento['Valor_num'].dtype)

In [ ]:
import pandas as pd
import numpy as np

#Armar una funcion que prepare los indicadores que necesito para el mapa
def preparar_indicadores_mapa(df):
  df = df.copy()
  df.columns = df.columns.str.strip() # print(df.columns.tolist()) #chequeo temporal

#Asegurarse que Valor_num sea numérico
df['Valor_num'] = pd.to_numeric(df['Valor_num'], errors='coerce')

# Columnas clave para agrupar
keys = [
    'id_punto_geografico',
    'Etapa',
    'Nombre del punto',
    'lat_wgs84',
    'lon_wgs84']

# Initialize salida with unique points and stages to avoid NameError
salida = df.drop_duplicates(subset=keys)[keys].copy()

# Unificar el nombre de la columna para usar de filtro para 'Metales Totales' and 'PAH Totales'
# 'Caracterización de la muestra' esta presente en suelo y sedimento pero en agua se llama 'Tipo de análisis'
if 'Caracterización de la muestra' in df.columns:
    analysis_col = 'Caracterización de la muestra'
elif 'Tipo de análisis' in df.columns:
    analysis_col = 'Tipo de análisis'
else:
     print("Warning: Neither 'Caracterización de la muestra' nor 'Tipo de análisis' found for metal/PAH filtering.")
    analysis_col = 'Parámetro'

#HTP total
htp_directo = (
    df[
        df['Parámetro']
        .astype(str)
        .str.contains('Hidrocarburos totales',
                      case=False, na=False)
    ]
    .groupby(keys)['Valor_num']
    .sum()
    .reset_index(name='HTP Total')
)
salida = salida.merge(htp_directo, on=keys, how='left')

#Metales totales
metales = (
    df[
        df[analysis_col]
       .astype(str)
       .str.contains(
           'Metales Totales',
           case=False, na=False)
       ]
    .groupby(keys)['Valor_num']
    .sum()
    .reset_index(name='Metales Totales')
)
salida = salida.merge(metales, on=keys, how='left')

# PAH totales
pah = (
    df[
        df[analysis_col]
          .astype(str)
          .str.contains(
              'PAH', case=False, na=False)
          ]
       .groupby(keys)['Valor_num']
       .sum()
       .reset_index(name='PAH Totales')
)
salida = salida.merge(pah, on=keys, how='left')

return salida

ACA QUEDE, HAY UN PROBLEMA PORQUE LOS DATASET TIENEN LA INFO EN DIFERENTES NOMBRES DE COLUMNAS

**Resumen del dataset:**

*- Dataset suelo:*

2 monitoreos, el primero es desde Ago-14 hasta Oct-19 (711) y el segundo Ene-19 hasta Sep-19 (288).

22 columnas originales con los nombre: 'Número de informe', 'Nombre de la Evaluación', 'Etapa', 'Componente ambiental', 'Procedencia de la Muestra', 'Procedencia Especifica de la Muestra', 'Nombre del punto', 'Este', 'Norte', 'Altitud', 'Zona', 'Datum', 'Descripción de ubicación',
'Tipo de muestra', 'Tipo de análisis', 'Etapa de extracción secuencial', 'Caracterización de la muestra', 'Fecha', 'Hora', 'Valor', 'Parámetro', 'Unidad de medida'.

**HACER LO MISMO PARA AGUA Y SEDIMENTO**

In [ ]:
## Análisis exploratorio de datos: Parámetros analizados por matriz
print('Caracterizacion fisicoquímica de la muestra: Matriz Suelo')
print(suelo['Caracterización de la muestra'].value_counts())

In [ ]:
print('Caracterizacion fisicoquímica de la muestra: Matriz Agua')
print(agua['Parámetro'].value_counts())

In [ ]:
print('Caracterizacion fisicoquímica de la muestra: Matriz Sedimento')
print(sedimento['Parámetro'].value_counts())

## 4.2 Estadística descriptiva

In [ ]:
# Estadistica Descriptiva General por parámetro y por muestreo
stats_suelo_M1 = (
    suelo[suelo['Etapa'] == 'Primer Monitoreo']
    .groupby('Parámetro')['Valor_num']
    .agg(
        n='count',
        min='min',
        max='max',
        media='mean',
        std='std',
        mediana = 'median')
    .round(4)
    .sort_values('media', ascending=False)
)

print('Estadisticas descriptivas por parámetro para el primero monitoreo de suelo (ordenadas por media):')
stats_suelo_M1

**AQUI QUEDÉ! TENGO QUE MODIFICAR EL CÓDIGO DE ARRIBA PARA PODER HACER LO MISMO PERO DIVIDIDO POR MUESTREO PORQUE CORRE RARO (sara)**

## 4.3 Comparación entre matrices

## 4.4 Detección de parámetros críticos

# 5. Exploración Normativa

*   Asignación de valores de refencia
*   Comparación de datos con Normativa Perú
*   Comparación de datos con Normativa Argentina
*   Cálculo de excedencias
*   Porcentaje de superación por parámetro y matriz









# 6. Análisis espacial

*   Georeferenciación de puntos ✅
- Exploracion de coordenadas (revisar si es repetido) ⏳
*   Visualización de puntos de monitoreo (por matriz) ✅ (Lo hice junto)
- Delimitación de AOI ⏳ (Está en 8.2)
*   Identificación de focos de contaminación
*   Revisión de coordenadas anómalas (¿Según AOI?)
*   Definición de área de influencia de afectación (opcional)







## 6.1 Importación de librerias de georreferenciación

In [ ]:
### Georeferenciación de puntos
# Importar librerías
!pip install folium --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import folium

%matplotlib inline
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size']   = 10

print("Listo")

## 6.2 Exploración de coordenadas

In [ ]:
### Coordenadas por Nombre del punto
## Suelo
# UTM
print("Ubicación suelo")
UTM_suelo = (
    suelo.groupby('Nombre del punto')
    .agg(
        lat=('Norte','first'),
        lon=('Este','first'),
        altitud=('Altitud','first'),
    )
    .round(2)
)

UTM_suelo

In [ ]:
## Suelo
# WGS84

WGS84_suelo = (
    suelo.groupby('Nombre del punto')
    .agg(
        lat=('lat_wgs84','first'),
        lon=('lon_wgs84','first'),
        altitud=('Altitud','first'),
    )
    .round(2)
)

WGS84_suelo

In [ ]:
### Coordenadas por Nombre del punto
## Agua
# UTM
print("Ubicación agua")
UTM_agua = (
    suelo.groupby('Nombre del punto')
    .agg(
        lat=('Norte','first'),
        lon=('Este','first'),
        altitud=('Altitud','first'),
    )
    .round(2)
)

UTM_agua

In [ ]:
## Agua
# WGS84
WGS84_agua = (
    agua.groupby('Nombre del punto')
    .agg(
        lat=('lat_wgs84','first'),
        lon=('lon_wgs84','first'),
        altitud=('Altitud','first'),
    )
    .round(2)
)

WGS84_agua

In [ ]:
### Coordenadas por Nombre del punto
## Sedimento
# UTM
print("Ubicación sedimento")
UTM_sedimento = (
    suelo.groupby('Nombre del punto')
    .agg(
        lat=('Norte','first'),
        lon=('Este','first'),
        altitud=('Altitud','first'),
    )
    .round(2)
)

UTM_sedimento

In [ ]:
## Sedimento
# WGS84
WGS84_sedimento = (
    sedimento.groupby('Nombre del punto')
    .agg(
        lat=('lat_wgs84','first'),
        lon=('lon_wgs84','first'),
        altitud=('Altitud','first'),
    )
    .round(2)
)

WGS84_sedimento

## 6.3 Conteo de valor por punto de muestreo

In [ ]:
suelo['Nombre del punto'].value_counts()

In [ ]:
agua['Nombre del punto'].value_counts()

In [ ]:
sedimento['Nombre del punto'].value_counts()

## 6.4 Cantidad de puntos de muestreo

In [ ]:
### Cantidad de Puntos de Monitoreo /"estaciones" en cada BD

print("Cantidad de 'Nombre del punto' únicos en suelo:")
display(suelo['Nombre del punto'].nunique())

print("\nCantidad de 'Nombre del punto' únicos en agua:")
display(agua['Nombre del punto'].nunique())

print("\nCantidad de 'Nombre del punto' únicos en sedimento:")
display(sedimento['Nombre del punto'].nunique())

## 6.5 Visualización de puntos de muestreo en el mapa del Perú

In [ ]:
#### MAPA DE PUNTOS DE MONITOREO (WGS84)

# Centro del mapa — Lima
mapa = folium.Map(location=[-12.05, -77.02], zoom_start=5,
                  tiles='CartoDB positron')

# Suelo
for idx, row in WGS84_suelo.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=5,
        color='brown',
        fill=True,
        fill_opacity=0.7,
        popup=folium.Popup(
            f"<b>Punto:</b> {idx}<br><b>Tipo:</b> Suelo<br>" +
            f"<b>Lat:</b> {row['lat']:.6f}<br>" +
            f"<b>Lon:</b> {row['lon']:.6f}<br>" +
            f"<b>Altitud:</b> {row['altitud']} m",
            max_width=200
        ),
        tooltip=f"{idx} (Suelo - Altitud: {row['altitud']} m)"
    ).add_to(mapa)

# Agua
for idx, row in WGS84_agua.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=5,
        color='blue',
        fill=True,
        fill_opacity=0.7,
        popup=folium.Popup(
            f"<b>Punto:</b> {idx}<br><b>Tipo:</b> Agua<br>" +
            f"<b>Lat:</b> {row['lat']:.6f}<br>" +
            f"<b>Lon:</b> {row['lon']:.6f}<br>" +
            f"<b>Altitud:</b> {row['altitud']} m",
            max_width=200
        ),
        tooltip=f"{idx} (Agua - Altitud: {row['altitud']} m)"
    ).add_to(mapa)

# Sedimento
for idx, row in WGS84_sedimento.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=5,
        color='yellow',
        fill=True,
        fill_opacity=0.7,
        popup=folium.Popup(
            f"<b>Punto:</b> {idx}<br><b>Tipo:</b> Sedimento<br>" +
            f"<b>Lat:</b> {row['lat']:.6f}<br>" +
            f"<b>Lon:</b> {row['lon']:.6f}<br>" +
            f"<b>Altitud:</b> {row['altitud']} m",
            max_width=200
        ),
        tooltip=f"{idx} (Sedimento - Altitud: {row['altitud']} m)"
    ).add_to(mapa)

# Leyenda manual
legend_html = '''
     <div style="position: fixed;
     bottom: 50px; left: 50px; width: 120px; height: 100px;
     border:2px solid grey; z-index:9999; font-size:14px;
     background-color:white;
     opacity:0.85;
     ">
       &nbsp; <b>Leyenda</b> <br>
       &nbsp; <i style="background:brown;opacity:0.85;width:10px;height:10px;display:inline-block;border-radius: 50%;"></i> Suelo <br>
       &nbsp; <i style="background:blue;opacity:0.85;width:10px;height:10px;display:inline-block;border-radius: 50%;"></i> Agua <br>
       &nbsp; <i style="background:yellow;opacity:0.85;width:10px;height:10px;display:inline-block;border-radius: 50%;"></i> Sedimento <br>
     </div>
     '''

mapa.get_root().html.add_child(folium.Element(legend_html))

display(mapa)

**Mini Conclusión:** Creo que por la concentración de puntos de muestreo podemos escoger la zona del norte del departamento de Piura (Talara)

**Mini conclusión:** VOY A ELEGIR PIURA O ESA ZONA

# 7. Análisis temporal (ES POSIBLE QUE NO SE HAGA DEBIDO A FRECUENCIA DE DATOS EN BDs)

*   Evolución temporal de HTP y otros compuestos de interes
*   Tendencia de contaminantes
*   Comparación entre monitoreos
*   STL (opcional)





# 8. Análisis de cobertura vegetal

*   Carga de Datos Landsat ✅
- Carga de Datos Sentinel ⏳
- Carga de Datos MapBiomas 😰
- Analisis de Bandas para cobertura Vegetal ⏳
- Calculo de NVDI
- Descripción general del sitio
*   Comparación temporal de la cobertura (3 puntos en el tiempo)
*   Relación entre los focos de contaminación y la cobertura vegetal
*   Identificación del ecosistema impactado





## 8.1 Carga de datos para cobertura vegetal: GEE (NDVI) + MapBiomas

Para realizar la comparación de los puntos de monitoreo con la cobertura vegetal, utilizaremos la plataforma Google Earth Engine (GEE). Los pasos principales serán:

1.  **Autenticación y Inicialización de GEE**: Conectarse a la API de Google Earth Engine.
2.  **Definir el Área de Interés (AOI)**: Establecer el área geográfica sobre la que se realizará el análisis.
3.  **Cargar imágenes satelitales (Sentinel-2)**: Utilizar imágenes para calcular el Índice de Vegetación de Diferencia Normalizada (NDVI).
4.  **Cargar datos de MapBiomas**: Acceder a la colección de MapBiomas Perú para obtener información sobre el uso y cobertura del suelo.
5.  **Calcular NDVI**: Implementar una función para obtener el NDVI de las imágenes.
6.  **Extraer información de vegetación para cada punto**: Obtener los valores de MapBiomas y NDVI en las coordenadas de los puntos de muestreo.

In [ ]:
# Instalar paquetes requeridos (ejecutar una sola vez)
!pip install geemap earthengine-api -q

# Importar la biblioteca Earth Engine
import ee
ee.Authenticate()

# Importar geemap para mapas interactivos
import geemap

# Otras bibliotecas que usaremos
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# INICIALIZACIÓN
# IMPORTANTE: ¡Debe coincidir exactamente con lo que se muestra en tu consola de Google Cloud!

ee.Initialize(project='ee-irabitsch') #utilizando proyecto de GEE de Ingrid Rabitsch

# Comprueba que funcionó
print("¡Earth Engine se inicializó correctamente!")
print(f"Versión de Earth Engine: {ee.__version__}")

## 8.2 Definición del Área de Interés (AOI)

Vamos a definir un Área de Interés que cubra la región de los puntos de muestreo en Perú. Podemos crear un polígono a partir de la extensión de los puntos WGS84 para asegurarnos de que cubra todas las áreas relevantes.

NOTA: COORDENADAS maximas de Piura con Buffer

In [ ]:
# Combinar todos los DataFrames WGS84 para obtener el rango geográfico completo
all_wgs84_points = pd.concat([WGS84_suelo[['lat', 'lon']],
                              WGS84_agua[['lat', 'lon']],
                              WGS84_sedimento[['lat', 'lon']]])

# POLÍGONO: Una forma personalizada definida por una lista de coordenadas
# El primer y el último punto deben ser iguales para cerrar el polígono
## (El departamento de Piura: 4º 5´ y 6º 22´ latitud sur, y 79º 00´ y 81º 7´ longitud oeste.) -4.08333, -6.36667, -79.00000, -81.11667
# Latitud: -4.08333 (norte), -6.36667 (sur)
# Longitud: -79.00000 (este), -81.50000 (oeste) (Ajustado)

min_lat = -6.36667
max_lat = -4.08333
min_lon = -81.50000
max_lon = -79.00000

# Crear un ee.Geometry.Rectangle como el Área de Interés (AOI)
aoi = ee.Geometry.Polygon([
    [min_lon, max_lat], # arriba izquierda
    [max_lon, max_lat], # arriba derecha
    [max_lon, min_lat], # abajo derecha
    [min_lon, min_lat], # abajo izquierda
    [min_lon, max_lat]  # cierre
])

# Añadir un buffer para asegurarse de que cubra todos los puntos y un poco más
BUFFER_DEG = 0.1 # Pequeño buffer en grados
aoi = ee.Geometry.Rectangle(
    [min_lon - BUFFER_DEG, min_lat - BUFFER_DEG, max_lon + BUFFER_DEG, max_lat + BUFFER_DEG]
)

print(f"AOI definida: Latitud {min_lat:.2f} a {max_lat:.2f}, Longitud {min_lon:.2f} a {max_lon:.2f}")

# Visualizar
map_aoi = folium.Map(location=[(min_lat + max_lat) / 2, (min_lon + max_lon) / 2], zoom_start=6,
                  tiles='CartoDB positron') # Modificación: se añadió el argumento tiles
folium.GeoJson(aoi.getInfo()).add_to(map_aoi)
display(map_aoi)


**NOTA:** Intento delimitar el AOI con FAO/GAUL de departamentos del Peru, pero no se puede

**OJO:** NO Resalta Piura!!!!

In [ ]:
# Cargar una FeatureCollection del catálogo GEE
# FAO GAUL (Global Administrative Unit Layers) — límites administrativos mundiales
departamentos_peru = (
    ee.FeatureCollection('FAO/GAUL/2015/level1')
    .filter(ee.Filter.eq('ADM0_NAME', 'Peru'))
)

total_deptos = departamentos_peru.size().getInfo()
print(f"Departamentos del Perú en GEE: {total_deptos}")

# Obtener un departamento específico — Piura
piura_depto = departamentos_peru.filter(
    ee.Filter.eq('ADM1_NAME', 'Piura')
).first()

props = piura_depto.getInfo()['properties']
print(f"\nPropiedades:")
print(f"  Nombre: {props['ADM1_NAME']}")
print(f"  País:   {props['ADM0_NAME']}")
print(f"  Código: {props['ADM1_CODE']}")

In [ ]:
piura_depto

In [ ]:
# Visualizar departamentos del Perú
Mapa_piura = geemap.Map(center=[-5, -80], zoom=7)

# Resaltar Piura
Mapa_piura.add_ee_layer(
    piura_depto, {'color': 'blue', 'fillColor': 'yellow', 'width': 5},
    )

Mapa_piura

In [ ]:
#### MAPA DE PUNTOS DE MUESTREO EN AOI (WGS84)

# Suelo
for idx, row in WGS84_suelo.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=5,
        color='brown',
        fill=True,
        fill_opacity=0.7,
        popup=folium.Popup(
            f"<b>Punto:</b> {idx}<br><b>Tipo:</b> Suelo<br>" +
            f"<b>Lat:</b> {row['lat']:.6f}<br>" +
            f"<b>Lon:</b> {row['lon']:.6f}<br>" +
            f"<b>Altitud:</b> {row['altitud']} m",
            max_width=200
        ),
        tooltip=f"{idx} (Suelo - Altitud: {row['altitud']} m)"
    ).add_to(map_aoi)

# Agua
for idx, row in WGS84_agua.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=5,
        color='blue',
        fill=True,
        fill_opacity=0.7,
        popup=folium.Popup(
            f"<b>Punto:</b> {idx}<br><b>Tipo:</b> Agua<br>" +
            f"<b>Lat:</b> {row['lat']:.6f}<br>" +
            f"<b>Lon:</b> {row['lon']:.6f}<br>" +
            f"<b>Altitud:</b> {row['altitud']} m",
            max_width=200
        ),
        tooltip=f"{idx} (Agua - Altitud: {row['altitud']} m)"
    ).add_to(map_aoi)

# Sedimento
for idx, row in WGS84_sedimento.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=5,
        color='yellow',
        fill=True,
        fill_opacity=0.7,
        popup=folium.Popup(
            f"<b>Punto:</b> {idx}<br><b>Tipo:</b> Sedimento<br>" +
            f"<b>Lat:</b> {row['lat']:.6f}<br>" +
            f"<b>Lon:</b> {row['lon']:.6f}<br>" +
            f"<b>Altitud:</b> {row['altitud']} m",
            max_width=200
        ),
        tooltip=f"{idx} (Sedimento - Altitud: {row['altitud']} m)"
    ).add_to(map_aoi)

# Leyenda manual
legend_html = '''
     <div style="position: fixed;
     bottom: 50px; left: 50px; width: 120px; height: 100px;
     border:2px solid grey; z-index:9999; font-size:14px;
     background-color:white;
     opacity:0.85;
     ">
       &nbsp; <b>Leyenda</b> <br>
       &nbsp; <i style="background:brown;opacity:0.85;width:10px;height:10px;display:inline-block;border-radius: 50%;"></i> Suelo <br>
       &nbsp; <i style="background:blue;opacity:0.85;width:10px;height:10px;display:inline-block;border-radius: 50%;"></i> Agua <br>
       &nbsp; <i style="background:yellow;opacity:0.85;width:10px;height:10px;display:inline-block;border-radius: 50%;"></i> Sedimento <br>
     </div>
     '''

map_aoi.get_root().html.add_child(folium.Element(legend_html))

display(map_aoi)

## 8.4 Cargar imágenes satelitales LANDASAT 8 y Sentinel-2 y calcular NDVI

Vamos a cargar la colección de imágenes LANDASAT y Sentinel-2, filtrarla por fecha y por el Área de Interés (AOI) que hemos definido. Luego, aplicaremos una función para calcular el NDVI.

In [ ]:
# Visualizar firmas espectrales típicas de coberturas peruanas
import numpy as np
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 6))

# Longitudes de onda de las bandas de Landsat (en micrómetros)
longitudes_onda = [0.48, 0.56, 0.65, 0.86, 1.61, 2.20]
nombres_bandas  = ['Azul', 'Verde', 'Rojo', 'NIR', 'SWIR1', 'SWIR2']

# Valores típicos de reflectancia (proporción de luz reflejada, 0–1)
bosque_amazonico = [0.03, 0.06, 0.03, 0.50, 0.28, 0.14]  # NIR alto, Rojo bajo
suelo_andino     = [0.12, 0.16, 0.22, 0.27, 0.32, 0.30]  # gradualmente creciente
lago_titicaca    = [0.05, 0.04, 0.03, 0.02, 0.01, 0.01]  # muy bajo en NIR
glacier_nevado   = [0.40, 0.42, 0.45, 0.50, 0.20, 0.12]  # alto en visible

ax.plot(longitudes_onda, bosque_amazonico, 'g-o',    lw=2, ms=10, label='Vegetación saludable')
ax.plot(longitudes_onda, suelo_andino, ls='-', marker='s',     lw=2, ms=10, color='brown', label='Suelo desnudo')
ax.plot(longitudes_onda, lago_titicaca,    'b-^',    lw=2, ms=10, label='Agua')
ax.plot(longitudes_onda, glacier_nevado, 'D', color="#E9E8E8", ls='-', mec='k', lw=2, ms=10, label='Glaciar/nieve')

# Sombrear regiones espectrales
ax.axvspan(0.45, 0.52, alpha=0.12, color='blue')
ax.axvspan(0.52, 0.60, alpha=0.12, color='green')
ax.axvspan(0.63, 0.69, alpha=0.12, color='red')
ax.axvspan(0.76, 0.90, alpha=0.12, color='darkred')

# Anotaciones para explicar características clave
ax.annotate('Clorofila\nabsorbe rojo\n(bosque sano)', xy=(0.65, 0.03), xytext=(0.50, 0.18),
            arrowprops=dict(arrowstyle='->', color='green'), fontsize=10, color='green')
ax.annotate('Alta reflectancia NIR\n(estructura foliar)', xy=(0.86, 0.50), xytext=(1.0, 0.45),
            arrowprops=dict(arrowstyle='->', color='green'), fontsize=10, color='green')

ax.annotate('Agua absorbse\nNIR/SWIR', xy=(1.0, 0.02), xytext=(1.2, 0.12),
            arrowprops=dict(arrowstyle='->', color='blue'), fontsize=10, color='blue')

ax.set_xlabel('Longitud de onda (µm)', fontsize=12)
ax.set_ylabel('Reflectancia', fontsize=12)
ax.set_title('Firmas Espectrales de Coberturas del Perú', fontsize=14)
ax.legend(loc='upper right', fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(0.4, 2.4)
ax.set_ylim(0, 0.6)

# Añadir etiquetas de bandas
for wl, name in zip(longitudes_onda, nombres_bandas):
    ax.annotate(name, (wl, 0.57), ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Cargar la colección de imágenes de LANDSAT 8
landsat_collection = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')

# Obtener la fecha de inicio más temprana de toda la colección
# Esto puede tomar un momento si la colección es muy grande
first_image_date = ee.Date(landsat_collection.first().get('system:time_start'))
print(f"Fecha de inicio de la colección LANDSAT 8: {first_image_date.format('YYYY-MM-dd').getInfo()}")


In [ ]:
###Crear Imagen

# 1. Cargar la colección de Landsat 8
landsat = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")

# 2. Definir un Bounding Box que envuelva a Piura
# Coordenadas aproximadas: [Oeste, Sur, Este, Norte]
piura_bbox = ee.Geometry.Rectangle([-82.00, -6.00, -80.00, -4.00]) #Latitud -6.37 a -4.08, Longitud -81.50 a -79.00

#3. Filtrar por un ciclo orbital exacto
ciclo_16_dias = (landsat
    .filterBounds(piura_bbox)
    .filterDate('2014-06-09', '2014-06-25') # Un ciclo completo de 16 días # 3. EL TRUCO: Filtrar por un ciclo orbital exacto # Esto garantiza que capturamos cada Path/Row del continente casi sin duplicados en el tiempo
)

# 4. Transformar las imágenes en polígonos limpios con sus datos de Path/Row
def extraer_cuadrícula(img):
    p = ee.Number(img.get('WRS_PATH'))
    r = ee.Number(img.get('WRS_ROW'))
    return ee.Feature(img.geometry(), {
        'PATH': p,
        'ROW': r,
        'Label': ee.String("P").cat(p.format("%d")).cat("R").cat(r.format("%d"))
    })

# Mapeamos la función y ejecutamos un .distinct() final por si acaso se solapan órbitas
cuadricula_piura = ciclo_16_dias.map(extraer_cuadrícula).distinct(['PATH', 'ROW'])

# 5. Crear el mapa interactivo centrado en el continente
Map = geemap.Map(center=[-5, -80], zoom=7)

# Estilo visual: Borde azul semi-transparente, sin relleno
estilo_capa = {
    'color': '#2b7bba',
    'fillColor': '00000000',
    'width': 1
}

# Añadir la cuadrícula completa al mapa
Map.addLayer(cuadricula_piura, estilo_capa, 'Cuadrícula Landsat Piura')

# Mostrar el mapa
Map

In [ ]:
# Extraer las propiedades 'PATH' y 'ROW' de cada Feature en cuadricula_piura
# y convertirlas a una lista de diccionarios.
path_row_list = cuadricula_piura.map(lambda f: ee.Feature(None, {
    'PATH': f.get('PATH'),
    'ROW': f.get('ROW')
})).getInfo()['features']

path_row_list

In [ ]:
imagen_piura = (landsat
    .filterDate('2014-06-09', '2014-06-25')   # época seca (julio-agosto) #'2014
    .filterBounds(cuadricula_piura)
    .filter(ee.Filter.inList('WRS_PATH', [10, 11]))     # Filtra el Path
    .filter(ee.Filter.inList('WRS_ROW', [63, 64]))     # Filtra el Row
    .filter(ee.Filter.lt('CLOUD_COVER', 30))   # < 20% nubes
    .sort('CLOUD_COVER')
    .first()
)

print(f"Bandas disponibles: {imagen_piura.bandNames().getInfo()[:8]}")

# Bandas
print("Landsat 8 bands:")
for band in imagen_piura.bandNames().getInfo():
    print(f"  {band}")

#### Bandas de Reflectancia Superficial (SR_B - Surface Reflectance Bands):

- SR_B1 (Aerosol Costero/Azul Ultra): Útil para estudios de calidad del agua costera, detección de aerosoles y vegetación de agua poco profunda.
- SR_B2 (Azul): Se usa para distinguir entre el suelo y la vegetación, estudios de calidad del agua y penetración atmosférica.
- SR_B3 (Verde): Refleja la vegetación sana y se utiliza en análisis de salud de la vegetación.
- SR_B4 (Rojo): Muy importante para la detección de vegetación, ya que la clorofila absorbe fuertemente en esta banda. Es clave para calcular el NDVI.
- SR_B5 (Infrarrojo Cercano - NIR): Alta reflectancia en vegetación sana debido a la estructura celular de las hojas. Fundamental para el cálculo de índices de vegetación como el NDVI.
- SR_B6 (Infrarrojo de Onda Corta 1 - SWIR 1): Útil para contenido de humedad en la vegetación y el suelo, discriminación de nubes y nieve.
- SR_B7 (Infrarrojo de Onda Corta 2 - SWIR 2): También sensible al contenido de humedad, rocas y minerales, y para diferenciar entre tipos de suelo.

#### Bandas de Calidad y Temperatura (ST - Surface Temperature & QA - Quality Assessment Bands):
- SR_QA_AEROSOL: Una banda de control de calidad para aerosoles, que ayuda a evaluar la corrección atmosférica y la presencia de partículas en el aire.
- ST_B10 (Temperatura de la Superficie de la Tierra - TIR 1): Esta es una banda térmica que se utiliza para estimar la temperatura de la superficie terrestre.
- ST_ATRAN: Transmitancia atmosférica. Indica cuánto de la radiación se transmite a través de la atmósfera, un factor clave en la corrección de temperatura.
- ST_CDIST: Distancia al centro de la nube. Puede ayudar a identificar píxeles cercanos a nubes para una mejor filtración.
- ST_DRAD: Radiación descendente. Cantidad de energía que llega a la superficie desde la atmósfera.
- ST_EMIS: Emisividad de la superficie. Mide la eficiencia con la que la superficie irradia energía térmica, variando según el material.
- ST_EMSD: Error estándar de la emisividad de la superficie.
- ST_QA: Control de calidad para la temperatura de la superficie. Proporciona información sobre la calidad y fiabilidad de la medición de la temperatura.
- ST_TRAD: Radiación ascendente. Cantidad de energía que emite la superficie.
- ST_URAD: Radiación ascendente atmosférica. Radiación emitida por la atmósfera hacia el sensor.
- QA_PIXEL: Una banda de control de calidad que identifica píxeles que pueden verse afectados por nubes, sombras de nubes, agua o nieve/hielo.
- QA_RADSAT: Indica si alguno de los detectores de la banda en el sensor está saturado, lo que puede afectar la precisión de los valores de reflectancia

In [ ]:
# Cargar la colección de imágenes de Sentinel-2
sentinel_collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')

# Obtener la fecha de inicio más temprana de toda la colección
# Esto puede tomar un momento si la colección es muy grande
first_image_date = ee.Date(sentinel_collection.first().get('system:time_start'))
print(f"Fecha de inicio de la colección Sentinel-2: {first_image_date.format('YYYY-MM-dd').getInfo()}")

In [ ]:
# Cargar la colección de imágenes de Sentinel-2
sentinel_collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')

# Obtener la fecha de fin más tardía de toda la colección
# Esto puede tomar un momento si la colección es muy grande
last_image_date = ee.Date(sentinel_collection.sort('system:time_end', False).first().get('system:time_end'))
print(f"Fecha de fin de la colección Sentinel-2: {last_image_date.format('YYYY-MM-dd').getInfo()}")

NOTA: Esto lo pongo porque no me entregaba informacion de NDVI porque eran muy antiguas.
Reporta:
Fecha de inicio de la colección Sentinel-2: 2015-07-04

In [ ]:
# Definir un rango de fechas para el análisis (ajustado para la disponibilidad de Sentinel-2)
start_date = '2015-07-04' ## FEcha de inicio de Sentinel / Primera Fecha de muestreo es 2013-10-16
end_date = '2019-12-31' ## FEcha del ultimo muestreo 2019-10-08

# Cargar la colección de imágenes Sentinel-2 L2A (nivel de procesamiento atmosféricamente corregido)
# Filtrar por fecha y por el AOI, y ordenar por la mínima nubosidad
sentinel_collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')\
    .filterDate(start_date, end_date)\
    .filterBounds(aoi)\
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))

# Aplicar la función NDVI a la colección de imágenes
ndvi_collection = sentinel_collection.map(calculateNDVI)

# Obtener la mediana de la colección NDVI para obtener una imagen compuesta sin nubes
ndvi_image = ndvi_collection.median().clip(aoi)

print(f"Imágenes Sentinel-2 filtradas para el periodo {start_date} a {end_date}.")
print("NDVI calculado y una imagen compuesta obtenida (mediana).")

## 8.5 Visualización del NDVI

Podemos visualizar el mapa de NDVI para tener una idea de la distribución de la vegetación en el Área de Interés.

In [ ]:
# Los datos de la Colección 2 de Landsat requieren escalado.
# Los valores brutos son enteros; necesitamos convertirlos a reflectancia (0-1).

# Aplicar factores de escala
def escalar_landsat(img):
    """
    Aplicar factores de escala a la reflectancia superficial de la Colección 2 de Landsat.
    La fórmula es: reflectancia = DN * 0,0000275 + (-0,2)
    Esto convierte los números enteros almacenados en valores de reflectancia reales.
    """

    bandas_opt = ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7']
    opt = img.select(bandas_opt)

    escalada = opt.multiply(0.0000275).add(-0.2)
    return img.addBands(escalada, overwrite=True)

imagen_escalada = escalar_landsat(imagen_piura)
print("Imagen escalada a valores de reflectancia (0–1)")

In [ ]:
# Visualizar imagen
# Color verdadero: Las bandas roja, verde y azul se muestran como RGB.
color_verdadero = {'bands': ['SR_B4', 'SR_B3', 'SR_B2'], 'min': 0, 'max': 0.3}

# Falso color: NIR, rojo y verde se muestran como RGB
# La vegetación aparece de un rojo brillante porque NIR se muestra como rojo
color_falso     = {'bands': ['SR_B5', 'SR_B4', 'SR_B3'], 'min': 0, 'max': 0.4}

Mapa3 = geemap.Map(center=[-4.5 , -81 ], zoom=8)
Mapa3.addLayer(imagen_escalada, color_verdadero, 'Color verdadero') #.clip(piura)
Mapa3.addLayer(imagen_escalada, color_falso, 'Color falso (vegetación = rojo)') #.clip(piura)
Mapa3

In [ ]:
# NDVI: Índice de Vegetación de Diferencia Normalizada
# # Fórmula: NDVI = (NIR - Rojo) / (NIR + Rojo)
# # Por qué funciona:
# - Vegetación sana: NIR ALTO, Rojo BAJO → NDVI cercano a +1
# - Suelo desnudo: NIR y Rojo similares → NDVI cercano a 0
# - Agua: NIR BAJO, Rojo ligeramente superior → NDVI negativo
#
# La normalización (dividir por la suma) permite comparar los resultados en
# diferentes condiciones de iluminación y calibraciones de sensores.

# Calcular índices espectrales
ndvi = imagen_escalada.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')


# Añadir índices a la imagen
imagen_con_indices = imagen_escalada.addBands([ndvi]) #, ndwi, nbr

# Visualizar NDVI
vis_ndvi = {
    'min': -0.2,
    'max': 0.9,
    'palette': ['#d73027', # Rojo: negativo (agua)
                '#fc8d59', # Naranja: bajo (suelo desnudo)
                '#fee08b', # Amarillo: moderado
                '#d9ef8b', # Verde claro: vegetación
                '#91cf60', # Verde medio: vegetación sana
                '#1a9850', # Verde oscuro: vegetación densa
                ]
}

Mapa4 = geemap.Map(center=[-4.5 , -81 ], zoom=8)
Mapa4.addLayer(ndvi.clip(imagen_escalada), vis_ndvi, 'NDVI')
Mapa4.add_colorbar(vis_ndvi, label='NDVI')
Mapa4

In [ ]:
# Otros índices útiles

# NDWI: Índice de Diferencia Normalizada del Agua
# Resalta las masas de agua (valores positivos = agua)
ndwi = imagen.normalizedDifference(['SR_B3', 'SR_B5']).rename('NDWI')

# NDBI: Índice de Diferencia Normalizada de Áreas Urbanizadas
# Resalta las áreas urbanas/desarrolladas (valores positivos = áreas urbanizadas)
ndbi = imagen.normalizedDifference(['SR_B6', 'SR_B5']).rename('NDBI')

# NDMI: Índice de Diferencia Normalizada de la Humedad
# Sensible al contenido de agua de la vegetación
ndmi = imagen.normalizedDifference(['SR_B5', 'SR_B6']).rename('NDMI')

# NBR: Índice de Areas quemadas
# Sensible a zonas quemadas
nbr  = imagen.normalizedDifference(['SR_B5', 'SR_B7']).rename('NBR')

# Agregar todos los índices a la imagen
imagen_con_índices = imagen.addBands([ndvi, ndwi, ndbi, ndmi, nbr])
print(f"Imagen ahora tiene {imagen_con_índices.bandNames().size().getInfo()} bandas")

In [ ]:
# Parámetros de visualización para NDVI
ndvi_vis_params = {
    'min': -1,
    'max': 1,
    'palette': ['red', 'orange', 'yellow', 'green', 'darkgreen'] # Paleta de colores para NDVI
}

# Añadir la capa NDVI al mapa de Folium del AOI
map_aoi.add_child(folium.raster_layers.ImageOverlay(
    image=ndvi_image.getThumbUrl(ndvi_vis_params),
    bounds=aoi.bounds().getInfo()['coordinates'][0][::2][::-1] + aoi.bounds().getInfo()['coordinates'][0][1::2][::-1],
    opacity=0.7,
    tooltip='NDVI'
))

# Añadir una leyenda para el NDVI (opcional, puede ser más compleja)
#folium.utilities.add_colormap(map_aoi, ndvi_vis_params, 'NDVI Range')

# Display the map
display(map_aoi)


## 8.6 Extraer información de vegetación para cada punto (NDVI)

Ahora que tenemos la imagen de NDVI, extraeremos los valores de NDVI para cada uno de tus puntos de monitoreo (suelo, agua y sedimento). Esto nos dará una métrica de la "salud" de la vegetación en cada ubicación.

In [ ]:
def extract_point_data(df_wgs84, ndvi_image, point_type):
    points = []
    for idx, row in df_wgs84.iterrows():
        # Crear un ee.Geometry.Point para cada punto
        point_geom = ee.Geometry.Point(row['lon'], row['lat'])

        try:
            # Obtener el valor de NDVI en el punto
            # reduceRegion es computacionalmente intensivo, se usa scale para definir la resolución
            ndvi_value = ndvi_image.sample(point_geom, scale=10).first().get('NDVI').getInfo()
            points.append({
                'Nombre del punto': idx,
                'lat_wgs84': row['lat'],
                'lon_wgs84': row['lon'],
                'Altitud': row['altitud'],
                'Tipo': point_type,
                'NDVI': ndvi_value
            })
        except Exception as e:
            print(f"Error al procesar el punto {idx} ({point_type}): {e}")
            points.append({
                'Nombre del punto': idx,
                'lat_wgs84': row['lat'],
                'lon_wgs84': row['lon'],
                'Altitud': row['altitud'],
                'Tipo': point_type,
                'NDVI': None # Indicar que no se pudo obtener el valor
            })
    return pd.DataFrame(points)

# Extraer NDVI para los puntos de suelo
ndvi_suelo_df = extract_point_data(WGS84_suelo, ndvi_image, 'Suelo')
print("NDVI extraído para puntos de Suelo:")
display(ndvi_suelo_df.head())

# Extraer NDVI para los puntos de agua
ndvi_agua_df = extract_point_data(WGS84_agua, ndvi_image, 'Agua')
print("\nNDVI extraído para puntos de Agua:")
display(ndvi_agua_df.head())

# Extraer NDVI para los puntos de sedimento
ndvi_sedimento_df = extract_point_data(WGS84_sedimento, ndvi_image, 'Sedimento')
print("\nNDVI extraído para puntos de Sedimento:")
display(ndvi_sedimento_df.head())


## 8.7 Cargar datos de MapBiomas y otros assets de GEE

Ahora, cargaremos los datos de cobertura y uso de suelo de MapBiomas Perú, así como cualquier otro asset específico de tu repositorio de Google Earth Engine si lo deseas. Asegúrate de tener los permisos adecuados para acceder a tus assets privados.

In [ ]:
import ee
ee.Authenticate()
ee.Initialize(project='ee-irabitsch')

mapbiomas = ee.ImageCollection("projects/mapbiomas-public/assets/peru/collection3/mapbiomas_peru_collection3_integration_v1");
print(mapbiomas);

In [ ]:
# Recortar AOI
mapbiomas_clip = mapbiomas.clip(aoi);

# 9. Integración de resultados

# 10. Visualizaciones finales - Organización de Dashboard

# 11. Conclusiones


*   Principales hallazgos
*   Respuestas a las preguntas de investigacion
*   Limitaciones del análisis
*   Recomendaciones para la Gestion Ambiental



# 12. Próximos pasos / Recomendaciones

*   Análisis estadísticos complementarios
*   Correlaciones
*   Dashboard interactivo